# 02 - Historical Trends: Blue Zone Countries vs the World

This notebook analyzes 60+ years of life expectancy trends, comparing Blue Zone countries
(USA, Japan, Italy, Greece, Costa Rica) against regional and global averages.

**Key questions:**
- How have Blue Zone countries performed relative to the global average over time?
- Is the Blue Zone advantage growing or shrinking?
- What are the decade-by-decade improvement rates?

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['figure.dpi'] = 100

PROJECT_DIR = os.path.dirname(os.path.abspath(os.getcwd()))
if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
    PROJECT_DIR = os.getcwd()
    if not os.path.exists(os.path.join(PROJECT_DIR, 'data')):
        PROJECT_DIR = os.path.dirname(PROJECT_DIR)

df = pd.read_csv(os.path.join(PROJECT_DIR, 'data', 'historical', 'merged_historical_panel.csv'))
bz_global = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'blue_zone_vs_global.csv'))
decades = pd.read_csv(os.path.join(PROJECT_DIR, 'outputs', 'analysis', 'decade_improvements.csv'))

print(f'Historical data: {len(df):,} rows, {df["iso_code"].nunique()} countries')
print(f'BZ vs Global: {len(bz_global)} years of comparison data')

Historical data: 5,952 rows, 93 countries
BZ vs Global: 64 years of comparison data


## 1. Blue Zone Countries vs Global Average Over Time

In [2]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), gridspec_kw={'height_ratios': [3, 1]})

# Top panel: Life expectancy lines
ax1.plot(bz_global['year'], bz_global['blue_zone_mean'], 'b-', linewidth=2.5,
         label='Blue Zone country average')
ax1.plot(bz_global['year'], bz_global['global_mean'], 'gray', linewidth=2,
         label='Global average', linestyle='--')
ax1.fill_between(bz_global['year'], bz_global['global_q25'], bz_global['global_q75'],
                 alpha=0.15, color='gray', label='Global IQR (25th-75th percentile)')

# Plot individual BZ countries
bz_colors = {'JPN': '#E74C3C', 'ITA': '#2ECC71', 'GRC': '#9B59B6', 'CRI': '#F39C12', 'USA': '#3498DB'}
bz_names = {'JPN': 'Japan', 'ITA': 'Italy', 'GRC': 'Greece', 'CRI': 'Costa Rica', 'USA': 'USA'}
for iso, color in bz_colors.items():
    c = df[(df['iso_code'] == iso) & (df['life_expectancy'].notna())]
    ax1.plot(c['year'], c['life_expectancy'], color=color, alpha=0.5, linewidth=1,
             label=bz_names[iso])

ax1.set_ylabel('Life Expectancy (years)', fontsize=12)
ax1.set_title('Blue Zone Countries vs Global Life Expectancy (1960-2023)', fontsize=14)
ax1.legend(loc='lower right', fontsize=9, ncol=2)
ax1.set_xlim(1960, 2023)

# Bottom panel: Gap
gap = bz_global['bz_gap_over_global'].dropna()
years = bz_global.loc[gap.index, 'year']
ax2.fill_between(years, 0, gap, alpha=0.3, color='blue')
ax2.plot(years, gap, 'b-', linewidth=1.5)
ax2.set_ylabel('BZ Gap (years)', fontsize=12)
ax2.set_xlabel('Year', fontsize=12)
ax2.axhline(y=0, color='black', linewidth=0.5)
ax2.set_title('Blue Zone Advantage Over Global Average', fontsize=11)
ax2.set_xlim(1960, 2023)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb02_bz_vs_global_detailed.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb02_bz_vs_global_detailed.png')

Saved: nb02_bz_vs_global_detailed.png


/tmp/ipykernel_1594754/4217381809.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Decade-by-Decade Comparison Table

In [3]:
# Show the key years comparison
key_years = bz_global[bz_global['year'].isin([1960, 1970, 1980, 1990, 2000, 2010, 2020])].copy()
display_cols = ['year', 'blue_zone_mean', 'global_mean', 'bz_gap_over_global', 'n_countries']
display_cols = [c for c in display_cols if c in key_years.columns]
key_years[display_cols].round(1)

,year,blue_zone_mean,global_mean,bz_gap_over_global,n_countries
0,1960,68.1,58.0,10.0,92
10,1970,70.8,61.7,9.0,93
20,1980,74.2,65.1,9.1,93
30,1990,76.8,67.8,9.0,93
40,2000,78.5,69.9,8.6,93
50,2010,80.6,73.1,7.6,93
60,2020,80.9,74.5,6.4,93


## 3. Improvement Rates by Decade

In [4]:
# Bar chart comparing BZ vs global improvement rates by decade
bz_decades = decades[decades['group'] == 'blue_zone'].copy()
global_decades = decades[decades['group'] == 'global'].copy()
non_bz_decades = decades[decades['group'] == 'non_blue_zone'].copy()

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(global_decades))
width = 0.25

bars1 = ax.bar(x - width, bz_decades['avg_gain'].values, width, label='Blue Zone countries',
               color='#2196F3', alpha=0.8)
bars2 = ax.bar(x, non_bz_decades['avg_gain'].values, width, label='Non-Blue Zone countries',
               color='#FF9800', alpha=0.8)
bars3 = ax.bar(x + width, global_decades['avg_gain'].values, width, label='Global average',
               color='#9E9E9E', alpha=0.8)

ax.set_xlabel('Decade', fontsize=12)
ax.set_ylabel('Average Life Expectancy Gain (years)', fontsize=12)
ax.set_title('Life Expectancy Gains per Decade: Blue Zone vs Rest of World', fontsize=14)
ax.set_xticks(x)
ax.set_xticklabels(global_decades['decade'].values)
ax.legend(fontsize=10)
ax.axhline(y=0, color='black', linewidth=0.5)

# Add value labels
for bars in [bars1, bars2, bars3]:
    for bar in bars:
        height = bar.get_height()
        if abs(height) > 0.1:
            ax.annotate(f'{height:.1f}',
                       xy=(bar.get_x() + bar.get_width() / 2, height),
                       xytext=(0, 3), textcoords='offset points',
                       ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb02_decade_improvements.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb02_decade_improvements.png')

Saved: nb02_decade_improvements.png


/tmp/ipykernel_1594754/2133112617.py:38: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Individual Blue Zone Country Trajectories

In [5]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

bz_countries = [('JPN', 'Japan - Okinawa'), ('ITA', 'Italy - Sardinia'),
                ('GRC', 'Greece - Ikaria'), ('CRI', 'Costa Rica - Nicoya'),
                ('USA', 'USA - Loma Linda')]

for i, (iso, name) in enumerate(bz_countries):
    ax = axes[i]
    c = df[(df['iso_code'] == iso) & (df['life_expectancy'].notna())].sort_values('year')
    
    # Country line
    ax.plot(c['year'], c['life_expectancy'], 'b-', linewidth=2, label=iso)
    
    # Global average
    ax.plot(bz_global['year'], bz_global['global_mean'], 'gray', linewidth=1,
            linestyle='--', alpha=0.7, label='Global avg')
    
    # Fill gap
    merged = c.merge(bz_global[['year', 'global_mean']], on='year', how='inner')
    ax.fill_between(merged['year'], merged['life_expectancy'], merged['global_mean'],
                    alpha=0.15, color='blue')
    
    ax.set_title(name, fontsize=12)
    ax.set_ylabel('LE (years)')
    ax.legend(fontsize=8)
    ax.set_xlim(1960, 2023)

# Remove empty subplot
axes[5].set_visible(False)

fig.suptitle('Individual Blue Zone Country Life Expectancy vs Global Average', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(PROJECT_DIR, 'outputs', 'figures', 'nb02_individual_bz_trajectories.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Saved: nb02_individual_bz_trajectories.png')

Saved: nb02_individual_bz_trajectories.png


/tmp/ipykernel_1594754/1391074812.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Key Findings

In [6]:
early = bz_global[bz_global['year'] <= 1965]
recent = bz_global[bz_global['year'] >= 2018]

if len(early) > 0 and len(recent) > 0:
    early_gap = early['bz_gap_over_global'].mean()
    recent_gap = recent['bz_gap_over_global'].mean()
    
    print('KEY FINDINGS:')
    print(f'1. Early 1960s Blue Zone gap: +{early_gap:.1f} years above global average')
    print(f'2. Recent (2018+) Blue Zone gap: +{recent_gap:.1f} years above global average')
    print(f'3. The gap has {"narrowed" if recent_gap < early_gap else "widened"} by {abs(early_gap - recent_gap):.1f} years')
    print(f'4. This suggests global {"convergence" if recent_gap < early_gap else "divergence"} - '
          f'the rest of the world is {"catching up" if recent_gap < early_gap else "falling behind"}')

# Best performing BZ country
latest = df[(df['is_blue_zone'] == 1) & (df['year'] >= 2020) & (df['life_expectancy'].notna())]
if len(latest) > 0:
    best = latest.groupby('iso_code')['life_expectancy'].mean().sort_values(ascending=False)
    print(f'\n5. Highest current LE among Blue Zone countries: {best.index[0]} ({best.iloc[0]:.1f} years)')
    print(f'6. Lowest current LE among Blue Zone countries: {best.index[-1]} ({best.iloc[-1]:.1f} years)')

KEY FINDINGS:
1. Early 1960s Blue Zone gap: +9.5 years above global average
2. Recent (2018+) Blue Zone gap: +6.2 years above global average
3. The gap has narrowed by 3.3 years
4. This suggests global convergence - the rest of the world is catching up

5. Highest current LE among Blue Zone countries: JPN (84.3 years)
6. Lowest current LE among Blue Zone countries: USA (77.3 years)
